# DIAMOND: Sterowanie Trajektorią w Celu Usunięcia Artefaktów w Modelach Flow Matching

![DIAMOND Diagram](img/diamond_diag.png)

Modele generatywne tworzące obrazy (takie jak FLUX czy SDXL) osiągnęły niesamowity poziom fotorealizmu. Niestety, wciąż borykają się z problemami anatomicznymi, strukturalnymi (np. wygenerowanie sześciu palców u dłoni, połamane kończyny) oraz poprawną reprezentacją tekstu na obrazie. 

Dotychczasowe metody naprawy tych błędów były inwazyjne – wymagały kosztownego dotrenowywania modeli (LoRA) albo ciężkich obliczeniowo poprawek już po wygenerowaniu obrazu (post-hoc).

Diamond (*Directed Inference for Artifact Mitigation in Flow Matching Models*) jest techniką kierowania trajektorią, który działa w trybie zero-shot i nie igneruje w model zajmujący się procesem generacji. 

Mechanizm działania polega na prognozowaniu wyglądu obiektu po odszumianiu na każdym kroku generacji, detekcje niechcianych artefaktów i edycję trajektorii w porządanym kierunku.

In [ ]:
!git clone https://github.com/gmum/DIAMOND
%cd DIAMOND

Cloning into 'DIAMOND'...
remote: Enumerating objects: 321, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 321 (delta 16), reused 2 (delta 0), pack-reused 288 (from 1)
Receiving objects: 100% (321/321), 769.29 MiB | 34.47 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/home/bartosz/Desktop/Studia/RepresentationDL/trajectory-guidence/DIAMOND


## Jak fizycznie działa DIAMOND?

DIAMOND działa w procesie odszumiania na zasadzie ciągłej oceny czystości obrazu (korekcja trajektorii krok po kroku). Proces nie jest ad-hoc, więc nie czeka do wygenerowania gotowego obrazu by go naprawić, składa się z:

1. Odtworzenie Czystego Obrazu "W Locie": Z modelu (szczególnie tych opartych o *Rectified Flow* takich jak FLUX, gdzie trajektorie są możliwie jak najprostsze/liniowe) wyciągamy predykcję prędkości wektora (velocity). Model oblicza, jak mógłby wyglądać zdekodowany i całkowicie wolny od szumu obraz docelowy $\hat{x}_{0,t}$ w aktualnym kroku $t$.
2. Detekcja Błędu: Szacunek przekazujemy do ogromnie ważnej części całego procesu - Detektora Artefaktów (Artifact Detector). Sieć ta skanuje obraz i weryfikuje za pomocą stworzonej maski, w którym miejscach pikseli powstanie artefakt (np. dodatkowy palec).
3. Obliczenie Błędu i Gradienty: Dzięki liniowej naturze transformaji FLUX (Rectified Flow), proces generowania opiera się na prostych, różniczkowalnych transformacjach. Pozwala to Detektorowi wyliczyć gradient błędu i skorygować trajektorię obrazu w czasie rzeczywistym. Sama strata jest z charakteru pixel wide, tzn. tylko obszary odpowiadające za niechciany artefakt zostaną zkorygowane.
4. Odpychanie Trajektorii: W naszym kroku *t* modyfikujemy bazowe pole, zachowując przy tym dynamiczność - kroki na początku są dużo ważniejsze niż pod koniec generacji, co znaczy, że podczas powstawania szkieletu obiektu zmiany trajektorii będą dużo silniejsze. Wartośc naszego kroku zależy więc od czasu i gradientów.


> *Dla przypomnienia z poprzedniej części: Dyfuzja oparta o tzw. Flow-Matching pozwala łączyć startowy próbkowany szum $x_1$ i wygenerowany ostatecznie obraz $x_0$ całkowicie liniowymi drogami. Sprawia to, że możemy wyliczyć predykcję końcową szybciej i prościej.*


![Flow Klatek](img/diamond_bball_sign.png)

### Core matematyki w projekcie: Liniowe przesuwanie trajektorii (Rectified Flow)

Istotą Flow Matchingu jest to, że staramy się uczyć ścieżek transportujących dane z szumu $x_1$ do obrazu docelowego $x_0$ prostymi liniami:
$$x_t = (1 - t)x_0 + t x_1$$
Dzięki relacjom matematycznym tej ścieżki i tzw. "prędkości" $v_\theta(x_t, t)$, model przewiduje docelowy, czysty obraz (clean data latent) w każdym dowolnym kroku czasowym podczas tworzenia go na samym "żywym" szumie:

$$\hat{x}_{0,t} = x_t - t \cdot v_\theta(x_t, t)$$

Głównym elementem "naprawy w locie" jest modyfikacja kroku obliczeniowego - całkowania Eulera, którego używa solver modelu dodając do niej wektor kierunkowy przeskalowany odpowiednim suwakiem skali $s_t$:

$$ x_{t-\Delta t} = x_t - \Delta t \cdot \left[ v_\theta(x_t,t) + s_t \cdot \nabla_{x_t}\mathcal{L}_{artifact}(\hat{x}_{0,t}) \right]$$

---

![Gradient normalization impact](img/diamond_norm_grad.png)

---

## Kluczowe Cechy Metodologii DIAMOND

Aby powyższy proces działał stabilnie potrzebne są:

1. **Normalizacja Gradientu (Gradient Normalization):**
Naiwne odjęcie wektora $\nabla_{x_t} \mathcal{L}_a$ powoduje, że norma gradientu bardzo rośnie, jeśli na docelowym obrazku znajduje się wiele artefaktów lub sam artefakt jest duży, dokonamy gwałtownej zmiany na obrazku, skutkującej w błędnym wyniku.
Zabieg optymalizujący sprowadza się do usunięcia "siły" gradientu za pomocą stałego wektora:
$$ \delta_t = \lambda_t \frac{\nabla_{x_t} L_a}{||\nabla_{x_t} L_a||_2 + \epsilon} $$
Oddzielenie siły (określanej od razu przez skalar przypisany do kroku generacji $\lambda_t$) od kierunku gradientu upewnia nas w tym, że proces zawsze pozostanie spójny, niezaleznie od rozmiaru korygowanej cechy, jedyne co wpływa na siłę to $\lambda_t$.

2. **Skalowanie w czasie (Power Schedule, $\lambda_t$):**
Podczas pracy nad każdym krokiem w dyfuzji, początkowo generowany jest zarys kształtów globalnych, a detale (włosy, cienie, ostrości) zachodzą blisko końca tworzenia (**high-frequency details**).
Artefakty takie jak "dziwne ułożenie palców", "dodatkowa liczba kończyn" lub "niewyraźne/zepsute litery" to problemy strukturalne - powstają w już na początku. Zależność parametrów naprawy DIAMOND ustala się za pomocą funkcji wielomianowej, siła maleje potęgowo wraz z każdym kolejnym krokiem:
$$\lambda_t = \lambda_{end} + (\lambda_{start} - \lambda_{end}) \left( 1 - \frac{i}{N - 1} \right)^p$$


---

### Zadanie [Obowiązkowe] - Odtworzenie korekcji pojedynczego kroku trajektorii w DIAMOND
Wiesz już, że przewagą modelu DIAMOND jest zdolność do naprawy trajektorii w jej trakcie (inference-time) przy użyciu mechanizmu *Artifact Detector*. System prognozuje końcowy wynik na danym kroku $t$, weryfikuje jego dokładność i odpycha generację od niechcianych defektów obliczając spadki (gradienty).

Twój cel to ułożenie prostego szablonu jednej iteracji aktualizującej trajektorie obrazka (z wektora $\hat{x}_{0,t}$). Musisz wypełnić brakujące części algorytmu (miejsca oznaczone jako `???`) w funkcji `step_diamond`.

**Co musisz zrobić?**
1. Odtworzyć szacowany obraz $\hat{x}_{0,t}$ za pomocą udostępnionego wektora wyjściowego i odjętego fragmentu przebytego *czasu* i zmierzonej prędkosci $\dots - t \cdot v_\theta(x_t,t)$.
2. Ocenić artefaktową "szkodliwość" tego czystego obrazka używająć specjalnie przekierowanego modułu `artifact_detector`.
3. Dodaj gradient naprawczy `grad_x` w krok relokujący zmienną stanu (metoda odejmowania ułamkowego *dt* i odrzucająca dany artefaktowy wskaźnik) do `x_t_minus_dt`.


In [2]:
import torch
import torch.nn as nn
# To jest uproszczony pseudokod ilustrujący pojedynczy krok wnioskowania w modelu DIAMOND

class DummyArtifactDetector(nn.Module):
    def forward(self, clean_image_estimate):
        # Symulacja błędu (Loss) dla artefaktów (np. 6 palców) na czystym obrazie
        # Im wyższa wartość funkcji straty, tym gorzej dla wygenerowanego obrazu
        loss = torch.sum(clean_image_estimate ** 2) 
        return loss

class DummyFlowModel:
    def __init__(self):
        self.artifact_detector = DummyArtifactDetector()
        
    def get_velocity(self, x_t, t):
        # Przewidywanie 'prędkości' u trajektorii modelu przez sieć układu (Flow Matching)
        # v_theta(x_t, t)
        return torch.randn_like(x_t)
        
    def step_diamond(self, x_t, t, dt, guidance_scale=1.0):
        # 1. Wymagamy by środowisko śledziło gradienty względem x_t (potrzebne do odpychania trajektorii)
        x_t = x_t.clone().detach().requires_grad_(True)
        
        # 2. Predykcja wektora prędkości v_theta(x_t, t) z zamrożonego wagi (bazowego modelu przepływu)
        v = self.get_velocity(x_t, t)
        
        # 3. Wytworzenie predykcji CZYSYTEGO obrazu (tzw. clean image estimate) na kroku czasu t
        # Wzór matematyczny: x_estimate = x_t - t * v
        x_estimate = x_t - t * v
        
        # 4. Ewaluacja błędu punktowego (Artifact Loss) na naszym "zapoznanym", czystym obrazie x_estimate
        # Musisz wrzucić czysty obraz w przygotowany detektor artefaktów by odkrył nieprawidłowości (zwrócił błąd loss)
        loss = self.artifact_detector(x_estimate)
        
        # 5. Obliczenie odpychającego gradientu (wektor odsuwający nas od wykrytego artefaktu)
        grad_x = torch.autograd.grad(loss, x_t)[0]
        
        # Sekretem metodologii DIAMOND jest normalizacja gradientu! 
        # Niezależnie czy robimy mały czy wielki poprawiany detal, kierunek pozostanie niezaburzony.
        grad_norm = grad_x / (torch.norm(grad_x) + 1e-8)
        
        # 6. Aktualizacja trajektorii modelu (Linear- korygujący krok modyfikowany o znormalizowany gradient naprawczy DIAMOND z siłą guidance_scale)
        # Wzór: x_{t - dt} = x_t - dt * ( v + s_t * \nabla_x Loss / ||\nabla_x Loss|| )
        x_t_minus_dt = x_t - dt * ( v + guidance_scale * grad_norm )
        
        return x_t_minus_dt.detach()

# --- Inicjalizacja i wykonanie instrukcji w teście ---
sim_model = DummyFlowModel()

# Losowy szum stanu obrazu (Latent x_t) dany w czasie t=0.5 z modelu generatywnego
current_latent_x_t = torch.randn(1, 4, 64, 64) 
t_current = 0.5
time_step_dt = 0.05
scale_of_correction = 5.0 # (s_t) - siła naprawcza detektora DIAMOND (guidance scale)

# Wykonaj zaktualizowany jednorazowy krok naprawczy DIAMOND na przebytej trajektorii
next_latent_step = sim_model.step_diamond(
    x_t=current_latent_x_t, 
    t=t_current, 
    dt=time_step_dt, 
    guidance_scale=scale_of_correction
)

print("Kolejna iteracja ukrytego 'czystego' po zredukowaniu krokem z korektą DIAMOND wykonana! Wymiary:", next_latent_step.shape)

Kolejna iteracja ukrytego 'czystego' po zredukowaniu krokem z korektą DIAMOND wykonana! Wymiary: torch.Size([1, 4, 64, 64])
